In [ ]:
# 1/4 - clone both: TokenVector main (CPU native) + cuda branch (linear_dw.tkv)
import subprocess, os
print('===== NATIVE MIGRATION BIT-EXACT TEST =====')
for d,br in [('/content/TVmain','main'),('/content/TVcuda','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build CPU native .so (tkv_simd_native.c) + CUDA kernel .so from .tkv-generated .cu
import subprocess, shutil, os, glob
root='/content/TVmain/Release/tools/simd_native'
gcc=shutil.which('gcc') or 'gcc'
# CPU native: build with gcc (needs -lm -lpthread -ldl)
srcs='tkv_simd_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_attn_cuda.c tkv_cuda_native.c tkv_linear_dx.c tkv_prof.c'
p=subprocess.run('cd '+root+' && '+gcc+' -O2 -mavx2 -mfma -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('CPU native build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:3000])
so_cpu=root+'/tkv_simd_native.so'
print('cpu so exists',os.path.exists(so_cpu))
# CUDA kernel from .tkv (pre-generated .cu, no Windows exe needed)
ex='/content/TVcuda/Release/cuda_backend/examples'
cu=ex+'/linear_dw.cuda.cu'; so_cuda=ex+'/linear_dw.so'
nvcc=shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
p2=subprocess.run(nvcc+' -O2 -shared -std=c++20 --expt-relaxed-constexpr -x cu -Xcompiler -std=c++20 -Xcompiler -fPIC '+cu+' -o '+so_cuda+' -lcudart', shell=True, capture_output=True, text=True, timeout=600)
print('CUDA kernel build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bit-exact compare: CPU LinearDwFused vs CUDA tkv kernel
import ctypes, numpy as np, torch
root='/content/TVmain/Release/tools/simd_native'
cpu=ctypes.CDLL(root+'/tkv_simd_native.so')
cpu.LinearDwFused.argtypes=[ctypes.POINTER(ctypes.c_float),ctypes.c_int,
                            ctypes.POINTER(ctypes.c_float),ctypes.c_int,
                            ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.c_int,
                            ctypes.c_int,ctypes.c_int,ctypes.c_int]
cpu.LinearDwFused.restype=ctypes.c_int
F32=ctypes.c_float; I32=ctypes.c_int; I64=ctypes.c_int64
cuda=ctypes.CDLL('/content/TVcuda/Release/cuda_backend/examples/linear_dw.so')
cuda.tkv_launch_linear_dw.argtypes=[I64,I64,I64,I32,I32,I32,I32,I32]; cuda.tkv_launch_linear_dw.restype=I32
rng=np.random.default_rng(7)
n=8; in_d=16; out_d=32
x=rng.standard_normal(n*in_d).astype(np.float32)
gy=rng.standard_normal(n*out_d).astype(np.float32)
g0=rng.standard_normal(in_d*out_d).astype(np.float32)
# CPU
gc=g0.copy()
rc=cpu.LinearDwFused(x.ctypes.data_as(ctypes.POINTER(F32)),x.size,
                     gy.ctypes.data_as(ctypes.POINTER(F32)),gy.size,
                     gc.ctypes.data_as(ctypes.POINTER(F32)),gc.size,0,n,in_d,out_d)
print('CPU rc=',rc)
# CUDA
xg=torch.from_numpy(x).cuda(); yg=torch.from_numpy(gy).cuda()
gg=torch.from_numpy(g0.copy()).cuda()
grid=(in_d*out_d+255)//256
rc2=cuda.tkv_launch_linear_dw(xg.data_ptr(),yg.data_ptr(),gg.data_ptr(),n,in_d,out_d,grid,256)
print('CUDA rc=',rc2)
a=gc.view(np.uint32); b=gg.cpu().numpy().view(np.uint32)
diff=np.count_nonzero(a!=b)
print('bitwise equal =',diff==0,' diff count=',diff,'/',a.size)
print('max abs diff =',np.abs(gc-gg.cpu().numpy()).max())

In [ ]:
# 4/4 - summary
print('If bitwise equal=False -> CUDA kernel KHONG the thay CPU native (loss EXACT 25.454).')